# 1. Data Exploration — NSL-KDD Intrusion Detection

**Goal of this notebook:** load the real NSL-KDD dataset, apply proper column names, and understand what we're working with before doing any modeling.

**Files needed:** place `KDDTrain+.txt` and `KDDTest+.txt` in the same folder as this notebook.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)

## Load the data

The raw `.txt` files have no header row — just comma-separated numbers. We apply the official 41 feature names (defined by the NSL-KDD creators) plus `label` and `difficulty`.

In [2]:
COLUMNS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root", "num_file_creations",
    "num_shells", "num_access_files", "num_outbound_cmds", "is_host_login",
    "is_guest_login", "count", "srv_count", "serror_rate", "srv_serror_rate",
    "rerror_rate", "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
    "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate", "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate", "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
    "label", "difficulty"
]

train = pd.read_csv("data/KDDTrain+.txt", names=COLUMNS)
test = pd.read_csv("data/KDDTest+.txt", names=COLUMNS)

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (125973, 43)
Test shape: (22544, 43)


**Expected output:** Train shape `(125973, 43)`, Test shape `(22544, 43)`. If your numbers differ, double check you downloaded the right files (not the 20% subset, unless that's intentional).

In [3]:
train.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,2,0.0,0.0,0.0,0.0,1.00,0.00,0.00,150,25,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,13,1,0.0,0.0,0.0,0.0,0.08,0.15,0.00,255,1,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,123,6,1.0,1.0,0.0,0.0,0.05,0.07,0.00,255,26,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,5,5,0.2,0.2,0.0,0.0,1.00,0.00,0.00,30,255,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,30,32,0.0,0.0,0.0,0.0,1.00,0.00,0.09,255,255,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


## Check for missing data

Always check this before anything else — NSL-KDD happens to be clean, but you should never assume that.

In [4]:
print("Missing values in train:", train.isnull().sum().sum())
print("Missing values in test:", test.isnull().sum().sum())

Missing values in train: 0
Missing values in test: 0


## Look at the attack labels

The `label` column has 39 distinct values in total across train+test — far more specific than we need. Let's see the raw distribution first.

In [5]:
print("Number of unique labels in train:", train["label"].nunique())
print("Number of unique labels in test:", test["label"].nunique())
print()
train["label"].value_counts()

Number of unique labels in train: 23
Number of unique labels in test: 38



label
normal             67343
neptune            41214
satan               3633
ipsweep             3599
portsweep           2931
smurf               2646
nmap                1493
back                 956
teardrop             892
warezclient          890
pod                  201
guess_passwd          53
buffer_overflow       30
warezmaster           20
land                  18
imap                  11
rootkit               10
loadmodule             9
ftp_write              8
multihop               7
phf                    4
perl                   3
spy                    2
Name: count, dtype: int64

**What to notice:** `normal` dominates with 67,343 rows. Some attack types like `spy` or `perl` have only a handful of examples. This imbalance is the central challenge of the whole project — keep it in mind.

## The most important check: novel attack types

The test set contains attack types that **never appear in the training set at all**. This is intentional — it simulates a real deployment scenario where new attacks show up that the model was never trained on.

In [6]:
train_labels = set(train["label"].unique())
test_labels = set(test["label"].unique())
novel_attacks = test_labels - train_labels

print(f"Attack types in test set that NEVER appear in training: {len(novel_attacks)}")
print(sorted(novel_attacks))

Attack types in test set that NEVER appear in training: 17
['apache2', 'httptunnel', 'mailbomb', 'mscan', 'named', 'processtable', 'ps', 'saint', 'sendmail', 'snmpgetattack', 'snmpguess', 'sqlattack', 'udpstorm', 'worm', 'xlock', 'xsnoop', 'xterm']


**Expected:** 17 novel attack types, including things like `apache2`, `mscan`, `httptunnel`, `worm`, `sqlattack`.

Keep this list — we'll use it later to measure how well our model generalizes to attacks it's never seen.

## Quick look at feature distributions

Let's sanity-check a few numeric features to build intuition.

In [7]:
train[["duration", "src_bytes", "dst_bytes", "count", "serror_rate"]].describe()

,duration,src_bytes,dst_bytes,count,serror_rate
count,125973.00000,1.259730e+05,1.259730e+05,125973.000000,125973.000000
mean,287.14465,4.556674e+04,1.977911e+04,84.107555,0.284485
std,2604.51531,5.870331e+06,4.021269e+06,114.508607,0.446456
min,0.00000,0.000000e+00,0.000000e+00,0.000000,0.000000
25%,0.00000,0.000000e+00,0.000000e+00,2.000000,0.000000
50%,0.00000,4.400000e+01,0.000000e+00,14.000000,0.000000
75%,0.00000,2.760000e+02,5.160000e+02,143.000000,1.000000
max,42908.00000,1.379964e+09,1.309937e+09,511.000000,1.000000


In [8]:
print("Protocol types:", train["protocol_type"].unique())
print("Flag values:", train["flag"].unique())
print("Number of distinct services:", train["service"].nunique())

Protocol types: ['tcp' 'udp' 'icmp']
Flag values: ['SF' 'S0' 'REJ' 'RSTR' 'SH' 'RSTO' 'S1' 'RSTOS0' 'S3' 'S2' 'OTH']
Number of distinct services: 70


## Save for the next notebook

We'll pass these loaded DataFrames to the next notebook by saving them as CSVs.

In [9]:
train.to_csv("data/nsl_kdd_train_raw.csv", index=False)
test.to_csv("data/nsl_kdd_test_raw.csv", index=False)
print("Saved.")

Saved.


## What's next

Open **`02_preprocessing.ipynb`** to group the 39 attack labels into 4 standard categories (DoS, Probe, R2L, U2R) and encode the text columns into numbers.